In [ ]:
from pathlib import Path
import pickle

import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

In [ ]:
DATASET_NAMES = ["sonar", "ionosphere", "wine"]
N_OUTER_SPLITS = 5
N_INNER_SPLITS = 5
RANDOM_STATE = 42
ARTIFACT_DIR = Path("artifacts")
DATASET_PATH = ARTIFACT_DIR / "openml_prepared_datasets.pkl"
SPLIT_PATH = ARTIFACT_DIR / "nested_cv_splits.pkl"

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

def load_and_prepare_dataset(dataset_name: str) -> dict:
    bunch = fetch_openml(name=dataset_name, version=1, as_frame=True)

    features = bunch.data.copy()
    target = bunch.target.copy()

    numeric_features = features.select_dtypes(include=[np.number]).copy()
    dropped_columns = [column for column in features.columns if column not in numeric_features.columns]

    if numeric_features.shape[1] == 0:
        raise ValueError(f"No numerical features were found for dataset '{dataset_name}'.")

    label_encoder = LabelEncoder()
    encoded_target = label_encoder.fit_transform(target.astype(str))

    return {
        "name": dataset_name,
        "openml_id": getattr(bunch, "details", {}).get("id", None),
        "target_name": getattr(bunch.target, "name", "target"),
        "X": numeric_features,
        "y": encoded_target,
        "label_encoder": label_encoder,
        "feature_names": numeric_features.columns.tolist(),
        "dropped_columns": dropped_columns,
        "n_samples": numeric_features.shape[0],
        "n_features": numeric_features.shape[1],
        "class_names": label_encoder.classes_.tolist(),
    }

def build_nested_cv_indices(y: np.ndarray, n_outer_splits: int = N_OUTER_SPLITS, n_inner_splits: int = N_INNER_SPLITS, random_state: int = RANDOM_STATE) -> dict:
    outer_splitter = StratifiedKFold(n_splits=n_outer_splits, shuffle=True, random_state=random_state)
    nested_splits = []

    for outer_fold, (outer_train_idx, outer_test_idx) in enumerate(outer_splitter.split(np.zeros(len(y)), y), start=1):
        inner_splitter = StratifiedKFold(n_splits=n_inner_splits, shuffle=True, random_state=random_state)
        inner_splits = []

        for inner_fold, (inner_train_rel_idx, inner_val_rel_idx) in enumerate(inner_splitter.split(np.zeros(len(outer_train_idx)), y[outer_train_idx]), start=1):
            inner_splits.append({
                "fold": inner_fold,
                "train_idx": outer_train_idx[inner_train_rel_idx],
                "val_idx": outer_train_idx[inner_val_rel_idx],
            })

        nested_splits.append({
            "fold": outer_fold,
            "train_idx": outer_train_idx,
            "test_idx": outer_test_idx,
            "inner_splits": inner_splits,
        })

    return {
        "n_outer_splits": n_outer_splits,
        "n_inner_splits": n_inner_splits,
        "random_state": random_state,
        "outer_splits": nested_splits,
    }


def load_and_prepare_text_dataset(file_path: str, dataset_name: str) -> dict:
    df = pd.read_excel(file_path)
    df = df.dropna(subset=['Posts', 'Labels'])

    vectorizer = TfidfVectorizer(max_features=500, stop_words='english')
    X_sparse = vectorizer.fit_transform(df['Posts'])
    X_df = pd.DataFrame(X_sparse.toarray(), columns=vectorizer.get_feature_names_out())

    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(df['Labels'].astype(str))

    return {
        "name": dataset_name,
        "openml_id": None,
        "target_name": "Labels",
        "X": X_df,
        "y": y_encoded,
        "label_encoder": label_encoder,
        "feature_names": X_df.columns.tolist(),
        "dropped_columns": [col for col in df.columns if col not in ['Posts', 'Labels']],
        "n_samples": X_df.shape[0],
        "n_features": X_df.shape[1],
        "class_names": label_encoder.classes_.tolist(),
    }

In [ ]:
prepared_datasets = {}
nested_cv_splits = {}

for dataset_name in DATASET_NAMES:
    dataset_payload = load_and_prepare_dataset(dataset_name)
    nested_cv_splits[dataset_name] = build_nested_cv_indices(dataset_payload["y"])
    prepared_datasets[dataset_name] = dataset_payload

text_payload = load_and_prepare_text_dataset("stress_data.xlsx", "stress_classification")
nested_cv_splits["stress_classification"] = build_nested_cv_indices(text_payload["y"])
prepared_datasets["stress_classification"] = text_payload

with DATASET_PATH.open("wb") as dataset_file:
    pickle.dump(prepared_datasets, dataset_file)

with SPLIT_PATH.open("wb") as split_file:
    pickle.dump(nested_cv_splits, split_file)

for dataset_name, payload in prepared_datasets.items():
    print(
        f"{dataset_name:21s} | X shape: {payload['X'].shape} | classes: {payload['class_names']} | saved split folds: {len(nested_cv_splits[dataset_name]['outer_splits'])}"
    )

print(f"Saved datasets to: {DATASET_PATH.resolve()}")
print(f"Saved split indices to: {SPLIT_PATH.resolve()}")